In [12]:
import os
import json
import re
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
import numpy as np
import pprint

from typing import List, Dict, Any, Tuple
from collections import defaultdict, Counter
from utils import count_relaxed_matches, count_exact_matches, exact_match_results, relax_match_results, sample_to_prediction_list


In [13]:
# model_folder = "Output_llama3.1_8b_no_sim"
model_folder = "Output_mistral-small_24b_no_sim"
# model_folder = "Output_gemma4_26b_no_sim"
# model_folder = "Output_gemma4_31b_no_sim"

In [14]:
if os.path.exists(f"results/Output_verification_em.csv") == True:
    os.remove(f"results/Output_verification_em.csv")
if os.path.exists(f"results/Output_verification_rm.csv") == True:
    os.remove(f"results/Output_verification_rm.csv")


tp_rm, fp_rm, fn_rm, od_rm = 0, 0, 0, 0
tp_em, fp_em, fn_em, od_em = 0, 0, 0, 0
for sample in os.listdir(model_folder):
    # if sample in ['sample_68', 'sample_635', 'sample_71', 'sample_324','sample_2748']:
        print(f"evaluating {sample}:")
        with open(f'label/{sample}.json', 'r') as f:
            label_dict = json.load(f)

        #Read output
        with open(f"{model_folder}/{sample}", 'r') as f:
            preds_dict = json.load(f)
        print(preds_dict)
        #Read original text
        with open(f'notes/{sample}.txt', 'r') as file:
            text = file.read()

        preds_dict_processed = []
        for ent in preds_dict:
            for phrase, label in ent.items():
        
                matches = list(re.finditer(re.escape(phrase), text,re.IGNORECASE))
                if matches:
                    for match in matches:
                        # print(f"Found '{match.group()}' at [{match.start()}, {match.end()}]")
                        preds_dict_processed.append({'entity':match.group(), 'label': label, 'start': match.start(), 'end': match.end()})
                else:
                    preds_dict_processed.append({'entity':phrase, 'label': 'OD', 'start': 0, 'end': 0})
                
        #Deduplication
        preds_dict_final = []
        seen = set()
        for d in preds_dict_processed:
            t = tuple(sorted(d.items()))
            if t not in seen:
                seen.add(t)
                preds_dict_final.append(d)
                
        ## Evaluation
        em, em_mm, em_ud, em_od = exact_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
        rm, rm_mm, rm_ud, rm_od = relax_match_results(label_dict, preds_dict_final, 'Output_verification', sample)
    
        tp_rm+= rm
        fp_rm+= rm_mm
        fn_rm+= rm_ud
        od_rm+= rm_od

            
        tp_em+= em
        fp_em+= em_mm
        fn_em+= em_ud
        od_em+= em_od

rm_precision = tp_rm/(tp_rm + rm_mm + od_rm)
rm_recall = tp_rm/(tp_rm + fn_rm)

em_precision = tp_em/(tp_em + em_mm + od_em)
em_recall = tp_em/(tp_em + fn_em)

print(f"SUMMARY EXACT MATCH: TP {tp_em}, FP {fp_em}, UD {fn_em}, OD {od_em}")
print(f"MICRO AVERAGE SCORE EXACT MATCH: Precision: {em_precision}, Recall: {em_recall}, F1 score: {2*em_precision*em_recall/(em_precision + em_recall)}")

print(f"SUMMARY RELAX MATCH: TP {tp_rm}, FP {fp_rm}, UD {fn_rm}, OD {od_rm}")
print(f"MICRO AVERAGE SCORE RELAX MATCH: Precision: {rm_precision}, Recall: {rm_recall}, F1 score: {2*rm_precision*rm_recall/(rm_precision + rm_recall)}")

evaluating sample_134_20210112211711:
[{'fever': 'PROBLEM'}, {'weight change': 'PROBLEM'}, {'fatigue': 'PROBLEM'}, {'aching': 'PROBLEM'}, {'fatigue': 'PROBLEM'}, {'aching': 'PROBLEM'}, {'glaucoma': 'PROBLEM'}, {'cataracts': 'PROBLEM'}, {'retinopathy': 'PROBLEM'}, {'hearing': 'PROBLEM'}, {'balance': 'PROBLEM'}, {'runny nose': 'PROBLEM'}, {'sinus problems': 'PROBLEM'}, {'nosebleeds': 'PROBLEM'}, {'dental problems': 'PROBLEM'}, {'hoarseness': 'PROBLEM'}, {'difficulty swallowing': 'PROBLEM'}, {'sore throat': 'PROBLEM'}, {'angina': 'PROBLEM'}, {'MI': 'PROBLEM'}, {'irregular heartbeat': 'PROBLEM'}, {'heart murmurs': 'PROBLEM'}, {'bad heart valves': 'PROBLEM'}, {'palpitations': 'PROBLEM'}, {'swelling of feet': 'PROBLEM'}, {'high blood pressure': 'PROBLEM'}, {'orthopnea': 'PROBLEM'}, {'paroxysmal nocturnal dyspnea': 'PROBLEM'}, {'stress test': 'TEST'}, {'arteriogram': 'TEST'}, {'pacemaker implantation': 'TREATMENT'}, {'cough': 'PROBLEM'}, {'sputum': 'PROBLEM'}, {'shortness of breath': 'PROBLEM

In [15]:
df = pd.read_csv('results/Output_verification_rm.csv',header=None)
df.columns = ['file','RM','MM','UD','OD','Prec','Sens','F1']
print(df['F1'].mean())

df.sort_values(by='OD',ascending=False).head(10)

0.7724769784172663


,file,RM,MM,UD,OD,Prec,Sens,F1
45,sample_68,59,32,0,116,0.2850,1.0000,0.4436
104,sample_635,132,58,16,107,0.4444,0.8919,0.5933
18,sample_324,75,4,44,58,0.5474,0.6303,0.5859
125,sample_163_20210112211715,113,49,3,46,0.5433,0.9741,0.6975
112,sample_2762,167,43,9,43,0.6601,0.9489,0.7786
39,sample_64_20210112211703,105,39,24,36,0.5833,0.8140,0.6796
96,sample_82_20210604143826_Booma,232,20,21,34,0.8112,0.9170,0.8609
74,sample_66_20210604143822_Booma,86,4,10,33,0.6992,0.8958,0.7854
43,sample_2747,68,2,21,32,0.6667,0.7640,0.7120
47,sample_36_20210610044412_Hayden,61,9,8,30,0.6100,0.8841,0.7219


# Further analysis